# MohaNetLight — Entrenamiento en Google Colab

Este notebook configura e instala los 3 paquetes necesarios para entrenar
MohaNetLight en Google Colab con GPU gratuita (T4).

**Paquetes:**
- `clash-royale-engine` — Motor de simulación Arena 1
- `clash-royale-gymnasium` — Entorno Gymnasium con observaciones parciales
- `mohanetlight` — Red neuronal AlphaStar-lite + PPO + bots heurísticos

> **Importante:** Selecciona GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*

## 1. Verificar GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No se detectó GPU — ve a Entorno de ejecución → Cambiar tipo")

## 2. Clonar repositorios e instalar

In [ ]:
%%bash
# ── Clonar los 3 repositorios ──────────────────────────────────────
# Si los repos son privados, usa un token de acceso personal:
#   git clone https://<TOKEN>@github.com/DeepRoyale-APREF/cr-engine.git

GITHUB_ORG="DeepRoyale-APREF"

for repo in cr-engine cr-gym MohaNetlight; do
  if [ ! -d "$repo" ]; then
    echo "Clonando $repo..."
    git clone --depth 1 https://github.com/$GITHUB_ORG/$repo.git
  else
    echo "$repo ya existe, actualizando..."
    cd $repo && git pull && cd ..
  fi
done

echo "✅ Repositorios listos"

In [ ]:
%%bash
# ── Instalar en orden de dependencias ─────────────────────────────
# 1) cr-engine (sin dependencias externas pesadas)
pip install -e cr-engine/ -q

# 2) cr-gym (depende de cr-engine)
pip install -e cr-gym/ -q

# 3) MohaNetLight (depende de cr-engine + cr-gym + torch)
#    torch ya viene preinstalado en Colab con CUDA
pip install -e "MohaNetlight/[dev]" -q

echo "✅ Todos los paquetes instalados"

## 3. Verificar instalación

In [ ]:
from mohanetlight import MohaNetLight, ModelConfig, TrainingConfig

cfg = ModelConfig()
model = MohaNetLight(cfg)

print(f"Parámetros: {model.count_parameters():,}")
print(f"Dispositivo auto-detectado: {TrainingConfig().device}")

In [ ]:
# Verificar forward pass en GPU
import torch

device = TrainingConfig().device
model = model.to(device)
hidden = model.init_hidden(1)
hidden = (hidden[0].to(device), hidden[1].to(device))

# Observación dummy
scalars = torch.randn(1, 16, device=device)
troops = torch.randn(1, 100, 14, device=device).abs()
troop_mask = torch.zeros(1, 100, dtype=torch.bool, device=device)
troop_mask[0, :5] = True
cards = torch.randn(1, 4, 4, device=device).abs()
action_masks = {
    "strategy": torch.ones(1, 3, dtype=torch.bool, device=device),
    "card": torch.ones(1, 5, dtype=torch.bool, device=device),
    "tile_x": torch.ones(1, 18, dtype=torch.bool, device=device),
    "tile_y": torch.ones(1, 32, dtype=torch.bool, device=device),
}

output = model.act(scalars, troops, troop_mask, cards, action_masks, hidden)
print(f"Acciones: {output.actions}")
print(f"Valor: {output.value.item():.4f}")
print(f"Log-prob: {output.log_prob.item():.4f}")
print(f"\n✅ Forward pass exitoso en {device}")

## 4. Ejecutar tests

In [ ]:
!cd MohaNetlight && pytest tests/ -v --tb=short 2>&1

## 5. Entrenamiento PPO con Liga

Entrena MohaNetLight contra bots heurísticos con evaluación periódica.

In [ ]:
from mohanetlight.config import ModelConfig, TrainingConfig
from mohanetlight.training.trainer import LeagueTrainer

model_cfg = ModelConfig()

# Ajustar para sesión de Colab (~2h con T4)
train_cfg = TrainingConfig(
    total_timesteps=500_000,    # 500K pasos (reducir para prueba rápida)
    n_steps=2560,               # pasos por rollout (~1 partida completa)
    n_epochs=4,                 # épocas PPO por actualización
    batch_chunk_len=32,         # longitud de chunk para BPTT truncado
    lr=3e-4,
    frame_skip=3,               # 3 frames/step → 10 decisiones/s, partida ≈ 2400 steps
    eval_interval=10,           # evaluar cada 10 actualizaciones
    eval_matches_per_pair=4,    # partidos por par en evaluación
    checkpoint_interval=25,     # guardar modelo cada 25 actualizaciones
    log_dir="./logs/colab_run",
    # device se auto-detecta → 'cuda' en Colab
)

print(f"Dispositivo: {train_cfg.device}")
print(f"Actualizaciones totales: {train_cfg.total_timesteps // train_cfg.n_steps}")
print(f"Frame skip: {train_cfg.frame_skip} → ~{30 // train_cfg.frame_skip} decisiones/s")
print(f"Steps por partida (~240s): ~{int(240 * 30 / train_cfg.frame_skip)}")

In [ ]:
trainer = LeagueTrainer(
    model_cfg=model_cfg,
    train_cfg=train_cfg,
)

# ¡Entrenar!
trainer.train()

## 6. Descargar checkpoints

Los checkpoints se guardan en `./logs/colab_run/`.

In [ ]:
import os
from google.colab import files

log_dir = "./logs/colab_run"
checkpoints = [f for f in os.listdir(log_dir) if f.endswith(".pt")]
checkpoints.sort()

print(f"Checkpoints disponibles: {checkpoints}")

if checkpoints:
    # Descargar el último checkpoint
    latest = os.path.join(log_dir, checkpoints[-1])
    print(f"Descargando {latest}...")
    files.download(latest)
    
    # También descargar métricas
    metrics_path = os.path.join(log_dir, "metrics.json")
    if os.path.exists(metrics_path):
        files.download(metrics_path)